In [ ]:
!pip install chromadb sentence-transformers datasets tqdm pandas

In [ ]:
!pip install rank-bm25

In [1]:
import sys
import os

project_root = os.path.abspath("..")
sys.path.insert(0, project_root)

print("Project root:", project_root)
print("Files:", os.listdir(project_root))

Project root: /Users/sivapriyasugumaran/Medical_Debate_System
Files: ['.DS_Store', 'requirements.txt', '.ipynb_checkpoints', 'data', 'outputs', 'notebooks', 'src']


In [2]:
from src.retrieval.load_dataset import load_pubmedqa
from src.retrieval.chunk_documents import chunk_documents
from src.retrieval.create_embeddings import load_embedding_model, create_embeddings
from src.retrieval.vector_store import create_chroma_collection, store_documents
from src.retrieval.bm25_index import build_bm25_index
from src.retrieval.reranker import load_reranker
from src.retrieval.retrieve_evidence import retrieve_evidence_hybrid_reranked
from src.retrieval.evaluate import evaluate_queries

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset = load_pubmedqa(limit=200)
documents = chunk_documents(dataset)

print("Loaded dataset:", len(dataset))
print("Built documents:", len(documents))
print(documents[0])

Loaded dataset: 200
Built documents: 200
{'id': '21645374', 'question': 'Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?', 'context': 'Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not unde

In [4]:
model = load_embedding_model()
embeddings = create_embeddings(model, documents)

print("Embeddings created:", len(embeddings))
print("Embedding dimension:", len(embeddings[0]))

Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:08<00:00,  1.21s/it]

Embeddings created: 200
Embedding dimension: 768


In [5]:
bm25 = build_bm25_index(documents)
print("BM25 index ready")

BM25 index ready


In [6]:
reranker = load_reranker()
print("Reranker loaded")

Reranker loaded


In [7]:
collection = create_chroma_collection(
    collection_name="pubmedqa_reranker_final_v1",
    persist_directory="./chroma_store_reranker_final_v1"
)

if collection.count() == 0:
    store_documents(collection, documents, embeddings=embeddings)

print("Chroma collection ready")
print("Collection count:", collection.count())

Chroma collection ready
Collection count: 200


In [8]:
# Demo / qualitative examples only

queries = [
    "Does post-mastectomy radiotherapy reduce breast cancer recurrence?",
    "Is chronic oro-facial pain associated with psychological factors?",
    "Can syncope in infants be related to water-induced urticaria?",
    "Does cardiopulmonary resuscitation improve survival outcomes?",
    "Do police enforcement policies reduce traffic fatalities?"
]

for query in queries:
    print("\n==============================")
    print("QUERY:", query)

    results = retrieve_evidence_hybrid_reranked(
        collection=collection,
        model=model,
        bm25=bm25,
        documents=documents,
        reranker=reranker,
        query=query,
        top_k=3,
        candidate_k=10
    )

    for i, item in enumerate(results, 1):
        print(f"\nResult {i}")
        print("Hybrid Score:", item.get("hybrid_score"))
        print("Rerank Score:", item.get("rerank_score"))
        print("Question:", item["metadata"]["question"])


QUERY: Does post-mastectomy radiotherapy reduce breast cancer recurrence?

Result 1
Hybrid Score: 0.03278688524590164
Rerank Score: 6.599774360656738
Question: Should chest wall irradiation be included after mastectomy and negative node breast cancer?

Result 2
Hybrid Score: 0.03200204813108039
Rerank Score: 4.459359645843506
Question: Does radiotherapy of the primary rectal cancer affect prognosis after pelvic exenteration for recurrent rectal cancer?

Result 3
Hybrid Score: 0.031754032258064516
Rerank Score: -1.344844937324524
Question: Does immediate breast reconstruction compromise the delivery of adjuvant chemotherapy?

QUERY: Is chronic oro-facial pain associated with psychological factors?

Result 1
Hybrid Score: 0.03278688524590164
Rerank Score: 8.433991432189941
Question: Are reports of mechanical dysfunction in chronic oro-facial pain related to somatisation?

Result 2
Hybrid Score: 0.015625
Rerank Score: -9.139456748962402
Question: Therapeutic anticoagulation in the trauma

In [9]:
evaluation_samples = [
    {
        "query": doc["question"],
        "relevant_id": doc["id"]
    }
    for doc in documents[:50]
]

In [10]:
results_per_query = []

for item in evaluation_samples:
    query = item["query"]
    relevant_id = item["relevant_id"]

    results = retrieve_evidence_hybrid_reranked(
        collection=collection,
        model=model,
        bm25=bm25,
        documents=documents,
        reranker=reranker,
        query=query,
        top_k=5,
        candidate_k=10
    )

    retrieved_ids = [r["id"] for r in results]

    results_per_query.append(
        {
            "query": query,
            "relevant_id": relevant_id,
            "retrieved_ids": retrieved_ids,
        }
    )

In [13]:
metrics = evaluate_queries(results_per_query, k=5)
print("Evaluation Metrics:")
for k,v in metrics.items():
    print(k, ":", round(v,3))

Evaluation Metrics:
Recall@5 : 1.0
HitRate@5 : 1.0
MRR@5 : 1.0


In [12]:
results = retrieve_evidence_hybrid_reranked(
    collection=collection,
    model=model,
    bm25=bm25,
    documents=documents,
    reranker=reranker,
    query="What causes chest pain?",
    top_k=3,
    candidate_k=10
)

for i, item in enumerate(results, 1):
    print(f"\nResult {i}")
    print("ID:", item["id"])
    print("Source:", item["source"])
    print("Hybrid Score:", item.get("hybrid_score"))
    print("Rerank Score:", item.get("rerank_score"))
    print("Metadata:", item["metadata"])
    print("Text:", item["text"][:700])


Result 1
ID: 18243752
Source: hybrid + reranker
Hybrid Score: 0.016129032258064516
Rerank Score: -8.517365455627441
Metadata: {'question': 'Should chest wall irradiation be included after mastectomy and negative node breast cancer?', 'final_decision': 'maybe', 'meshes': 'Adult, Aged, Breast Neoplasms, Female, Humans, Mastectomy, Middle Aged, Neoplasm Recurrence, Local, Radiotherapy, Retrospective Studies', 'labels': 'PURPOSE, PATIENTS AND METHODS, RESULTS'}
Text: Question: Should chest wall irradiation be included after mastectomy and negative node breast cancer? Context: This study aims to evaluate local failure patterns in node negative breast cancer patients treated with post-mastectomy radiotherapy including internal mammary chain only. Retrospective analysis of 92 internal or central-breast node-negative tumours with mastectomy and external irradiation of the internal mammary chain at the dose of 50 Gy, from 1994 to 1998. Local recurrence rate was 5 % (five cases). Recurrence sit